# Free-Droid (Szabi) — v2 fine-tune az új datasethez

Vékony futtató: telepíti az Unsloth-ot, klónozza a repót, és a `training/finetune.py`-t hívja. Minden logika a `finetune.py` + `config.py`-ban (verziókövetett).

**Mi új a v1-hez képest:**
- Az A/B **lezárva → Llama** (aszimmetrikus hibrid: edge **Llama 3.2 3B**, cloud **Llama 3.1 8B**).
- Dataset **745 példa** (train 671 / val 74): tömör „székely góbés" persona, megerősített tool-calling (~17%), és új „válasz adott kontextusból" (RAG-grounding) kategória.
- `--preset gentle` (lr 5e-5, 1 epoch, r/alpha 8) a kis adathalmaz túltanulása ellen — **ne hajszold az alacsony loss-t**, a validációs halmazon mérj.

**Először: Runtime → Change runtime type → T4 GPU.**

In [ ]:
# 1. Unsloth telepítése (hivatalos Colab-installer — illeszti a torch/bnb/triton verziókat).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Repo (dataset + finetune.py), majd be a training/-be.
#    A v2 dataset (745 példa) + a Llama-defaultok legyenek a default branchen — előbb merge.
!git clone --depth 1 https://github.com/pits2022/free-droid.git
%cd free-droid/training

## Edge modell — Llama 3.2 3B (offline fallback)

A Pi 5-ön, teljesen offline futó modell. Q4_K_M (edge) + Q8_0 (cloud) GGUF export. Kimenet: `outputs/llama3.2-3b-v2/`.

In [ ]:
!python finetune.py --variant llama --preset gentle --tag v2

## Cloud modell — Llama 3.1 8B (a fő demó-agy, CPU-cloud)

A CAX31-en CPU-n futó nagyobb modell — a magyar/persona minőséget a méret hozza. A `llama8b` variáns T4-biztos (batch=1, grad_accum=8) és csak Q4_K_M-et exportál. Kimenet: `outputs/llama3.1-8b-v2/`.

In [ ]:
!python finetune.py --variant llama8b --preset gentle --tag v2

## Next

- Kimenetek: `training/outputs/<variant>-v2/` — `gguf-q4_k_m` (edge), `gguf-q8_0` (cloud), `lora-adapter`. Töltsd le (vagy push HF Hubra) — git-ignorálva.
- **Ollama:** `ollama create szabi -f Modelfile` (a `FROM` a kiválasztott GGUF-ra mutasson). Ügyelj a helyes `TEMPLATE`-re (Llama-3 fejléc) + stop tokenekre — az Unsloth GGUF-export NEM ágyazza be a chat sablont, e nélkül halandzsa + nincs leállás.
- **Pontozás:** `persona_benchmark.json` + `ertekelo_sablon.md` (6 dimenzió).
- **Kötelező red-team pass** (provokatív / offtopic kérdések) a demó előtt.
- A **tényeket** futásidőben a `freedroid.rag` BM25 retriever adja a promptba (`yotengrit_corpus.json`); a fine-tune personát/stílust tanít, nem tényeket.